# 🎤 Audio Classification (KWS) — From SavedModel to Embedded C

**Renesas RA8P1 Model Zoo — Tutorial 03**

---

This notebook walks through the complete pipeline for deploying a **Keyword Spotting (KWS)**
DS-CNN model onto a **Renesas RA8P1 MCU** (Cortex-M85 + Ethos-U55 NPU)
using the **MERA (RUHMI) compiler**.

```
MLCommons Tiny DS-CNN (SavedModel, pretrained on Speech Commands v0.02)
        │
        ▼  [TFLiteConverter]
  TFLite FP32  (kws_ref_model_FP32.tflite)
        │
        ▼  [TFLite PTQ with MFCC calibration data]
  TFLite INT8  (kws_ref_model_INT8.tflite)
        │
        ▼  [MERA compiler — Section 5]
  C source files  (embedded_c/src_mcu/ or src_mcu_npu/)
        │
        ▼
  Flash to RA8P1 board
```

---

### Model Details

| Field | Value |
|-------|-------|
| **Model** | DS-CNN (Depthwise Separable CNN) |
| **Task** | 12-class Keyword Spotting |
| **Dataset** | Google Speech Commands v0.02 |
| **Classes** | down, go, left, no, off, on, right, stop, up, yes, silence, unknown |
| **Input** | 1-second 16 kHz mono audio → MFCC features `[1, 49, 10, 1]` |
| **Output** | `[1, 12]` — softmax probabilities (12 classes) |
| **Source** | [MLCommons Tiny Benchmark](https://github.com/mlcommons/tiny/tree/master/benchmark/training/keyword_spotting) |

---

### Audio → MFCC Feature Extraction Pipeline

```
16 kHz mono WAV (1 second = 16,000 samples)
        │
        ▼  Normalise to [-1, 1]
        │
        ▼  STFT (frame=480, hop=320, Hann window) → 49 frames
        │
        ▼  Mel filterbank (40 bins, 20–4000 Hz)
        │
        ▼  Log-mel → DCT (keep first 10 coefficients)
        │
        ▼  Reshape to [1, 49, 10, 1]  (batch, frames, coeffs, channel)
  MFCC features  →  model input
```

---

### References
- 📖 [MLCommons Tiny — KWS Benchmark](https://github.com/mlcommons/tiny/tree/master/benchmark/training/keyword_spotting)
- 📖 [Google Speech Commands Dataset](https://arxiv.org/abs/1804.03209)
- 📖 [DS-CNN Paper (Zhang et al.)](https://arxiv.org/abs/1711.07128)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import pathlib, os

_nb_dir = pathlib.Path(os.getcwd())
_repo   = _nb_dir.parent if _nb_dir.name == "tutorials" else _nb_dir
if not (_repo / "audio").exists():
    for _p in _nb_dir.parents:
        if (_p / "audio").exists():
            _repo = _p
            break
    else:
        raise FileNotFoundError(
            f"❌ Could not find the Model-zoo repo root.\n"
            f"   Expected an 'audio/' directory in or above: {_nb_dir}\n"
            f"   Make sure you cloned the full repo and are running this notebook from the 'tutorials/' folder."
        )

_kws_root = _repo / "audio" / "audio_classification" / "key_word_spotting" / "python"
_req_txt  = _kws_root / "requirements.txt"

!pip install --upgrade pip
!pip install --upgrade ipywidgets

# ── 1. MERA (RUHMI) compiler wheel (install first) ───────────────────────────
MERA_WHL = "mera-2.6.0+pkg.4513-cp310-cp310-manylinux_2_27_x86_64.whl"
MERA_URL = f"https://raw.githubusercontent.com/renesas/ruhmi-framework-mcu/main/install/{MERA_WHL}"
!wget -q --show-progress -O "{MERA_WHL}" "{MERA_URL}" && pip install "{MERA_WHL}"

# ── 2. Project requirements (TF 2.21.0 — overrides MERA's TF 2.18 pin) ──────
!pip install -r "{_req_txt}"

# ── 3. Extra dependencies (build + visualization) ────────────────────────────
!pip install tensorflow-datasets matplotlib IPython

print("✅ Dependencies installed")

---
## ⚙️ Setup — Configuration

Set paths, audio constants, and MFCC parameters for the KWS DS-CNN model.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

NUM_CALIB_SAMPLES = 200         # calibration samples for MERA NPU quantization
FORCE_REBUILD     = True        # set True to regenerate models even if they exist

# ─────────────────────────────────────────────────────────────────────────────
import pathlib, os
import numpy as np

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # Force CPU — avoids GPU/PTX errors
import tensorflow as tf

# ─────────────────────────────────────────────────────────────────────────────
# PATHS  (auto-detected from notebook location)
# ─────────────────────────────────────────────────────────────────────────────
_nb_dir    = pathlib.Path(os.getcwd())
REPO_ROOT  = _nb_dir.parent if _nb_dir.name == "tutorials" else _nb_dir
if not (REPO_ROOT / "audio").exists():
    for _p in _nb_dir.parents:
        if (_p / "audio").exists():
            REPO_ROOT = _p
            break
    else:
        raise FileNotFoundError(
            f"❌ Could not find the Model-zoo repo root.\n"
            f"   Expected an 'audio/' directory in or above: {_nb_dir}\n"
            f"   Make sure you cloned the full repo and are running this notebook from the 'tutorials/' folder."
        )

KWS_ROOT   = REPO_ROOT / "audio" / "audio_classification" / "key_word_spotting"
MODEL_DIR  = KWS_ROOT / "python"
TFLITE_DIR = MODEL_DIR / "model"
TEST_DATA  = MODEL_DIR / "sample_audio"
EMBED_DIR  = KWS_ROOT / "embedded_c"

FP32_PATH  = TFLITE_DIR / "kws_ref_model_FP32.tflite"
INT8_PATH  = TFLITE_DIR / "kws_ref_model_INT8.tflite"

# ─────────────────────────────────────────────────────────────────────────────
# KWS LABELS  (12-class, must match training label map)
# ─────────────────────────────────────────────────────────────────────────────
KWS_LABELS = [
    "down", "go", "left", "no", "off", "on",
    "right", "stop", "up", "yes", "silence", "unknown",
]

# ─────────────────────────────────────────────────────────────────────────────
# AUDIO / MFCC CONSTANTS  (must match training pipeline)
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_RATE            = 16000
CLIP_DURATION          = 1.0
DESIRED_SAMPLES        = int(SAMPLE_RATE * CLIP_DURATION)  # 16000
WINDOW_SIZE_SAMPLES    = 480        # 30 ms
WINDOW_STRIDE_SAMPLES  = 320        # 20 ms
NUM_MEL_BINS           = 40
DCT_COEFFICIENT_COUNT  = 10
LOWER_EDGE_HZ          = 20.0
UPPER_EDGE_HZ          = 4000.0

# ─────────────────────────────────────────────────────────────────────────────
print(f"TensorFlow version : {tf.__version__}")
print(f"Repository root    : {REPO_ROOT}")
print(f"FP32 model         : {FP32_PATH}")
print(f"INT8 model         : {INT8_PATH}")
print(f"Test data          : {TEST_DATA}")

---
## 📥 Section 1 — Download & Build Models (Optional)

This section downloads the pretrained KWS DS-CNN **SavedModel** from
[MLCommons Tiny](https://github.com/mlcommons/tiny), downloads the **Speech Commands v0.02**
calibration dataset, and converts the model to TFLite (FP32 + INT8):

| Step | Operation | Details |
|------|-----------|---------|
| 1 | **Download SavedModel** | From MLCommons Tiny GitHub |
| 2 | **Convert → FP32 TFLite** | `TFLiteConverter.from_saved_model()` |
| 3 | **Download calibration dataset** | Speech Commands v0.02 validation split (auto-downloaded via `tensorflow_datasets`) |
| 4 | **Convert → INT8 TFLite** | Post-training quantization with balanced MFCC calibration data |

```
SavedModel  (MLCommons Tiny GitHub)
      │
      ▼  tf.lite.TFLiteConverter.from_saved_model()
  kws_ref_model_FP32.tflite
      │
      ▼  Download Speech Commands v0.02 validation WAVs
      │  Extract WAVs per keyword sub-folder
      │  Balanced sampling → MFCC representative dataset
      │
      ▼  TFLiteConverter + representative_dataset
  kws_ref_model_INT8.tflite
```

> ⏭️ **Skip this cell** if the pre-built models already exist in `python/model/`.
> The cell checks automatically and exits early if all files are present.

### Requirements for building from scratch
- `tensorflow`, `tensorflow-datasets` (for auto-downloading Speech Commands v0.02)
- `tqdm` (optional, for progress bars)
- ~1 GB disk space for the Speech Commands dataset

In [ ]:
import pathlib, os, sys, urllib.request, glob as _glob, wave as _wave
from tqdm.auto import tqdm

# ── Check if pre-built models already exist ───────────────────────────────────
_all_exist = FP32_PATH.exists() and INT8_PATH.exists()

if _all_exist and not FORCE_REBUILD:
    print("ℹ️  All pre-built models found — skipping download & conversion.")
    print(f"   FP32  : {FP32_PATH}")
    print(f"   INT8  : {INT8_PATH}")
    print(f"   (Set FORCE_REBUILD = True in the config cell to regenerate)")
else:
    if FORCE_REBUILD and _all_exist:
        print("🔄  FORCE_REBUILD enabled — regenerating models from scratch ...\n")
        FP32_PATH.unlink(missing_ok=True)
        INT8_PATH.unlink(missing_ok=True)
    else:
        print("⚠️  One or more models missing — building from scratch ...\n")

    # ── Install extra build dependencies ──────────────────────────────────────
    !pip install -q tensorflow-datasets tqdm

    SAVED_MODEL_DIR = TFLITE_DIR / "kws_ref_model"
    CALIB_WAV_DIR   = TFLITE_DIR / "calib_wavs"
    TFLITE_DIR.mkdir(parents=True, exist_ok=True)

    # ══════════════════════════════════════════════════════════════════════════
    # Step 1: Download SavedModel from MLCommons Tiny
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 60)
    print("  Step 1: Download KWS DS-CNN SavedModel")
    print("=" * 60)

    _BASE_RAW = (
        "https://github.com/mlcommons/tiny/raw/master/"
        "benchmark/training/keyword_spotting/trained_models/kws_ref_model"
    )
    _SM_FILES = {
        "saved_model.pb":                          f"{_BASE_RAW}/saved_model.pb",
        "variables/variables.data-00000-of-00001": f"{_BASE_RAW}/variables/variables.data-00000-of-00001",
        "variables/variables.index":               f"{_BASE_RAW}/variables/variables.index",
    }

    if (SAVED_MODEL_DIR / "saved_model.pb").exists():
        print(f"  ✅ SavedModel already exists: {SAVED_MODEL_DIR}")
    else:
        print(f"  Downloading SavedModel to {SAVED_MODEL_DIR} ...")
        for rel_path, url in tqdm(_SM_FILES.items(), desc="  Downloading", unit="file"):
            dest = SAVED_MODEL_DIR / rel_path
            dest.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, str(dest))
        print(f"  ✅ SavedModel downloaded")

    # ── Converter helper (v2 with v1 fallback) ────────────────────────────────
    def _get_converter(saved_model_dir):
        """Return a TFLite converter, falling back to the v1 API if v2 fails."""
        try:
            return tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
        except (AttributeError, Exception) as e:
            print(f"  ⚠️  v2 converter failed ({e}), using tf.compat.v1 fallback")
            return tf.compat.v1.lite.TFLiteConverter.from_saved_model(
                str(saved_model_dir))

    # ══════════════════════════════════════════════════════════════════════════
    # Step 2: Verify SavedModel & Convert → TFLite FP32
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print("  Step 2: Verify SavedModel & Convert → TFLite FP32")
    print("=" * 60)

    # Verify SavedModel loads (non-fatal — conversion handles this separately)
    try:
        _loaded = tf.saved_model.load(str(SAVED_MODEL_DIR))
        _sigs = list(_loaded.signatures.keys())
        print(f"  SavedModel loaded — signatures: {_sigs}")
        if "serving_default" in _loaded.signatures:
            _sig = _loaded.signatures["serving_default"]
            _inp_info = {k: v.shape.as_list() for k, v in _sig.structured_input_signature[1].items()}
            _out_info = {k: v.shape.as_list() for k, v in _sig.structured_outputs.items()}
            print(f"  Input  shapes: {_inp_info}")
            print(f"  Output shapes: {_out_info}")
    except Exception as e:
        print(f"  ⚠️  SavedModel verification skipped (non-fatal): {e}")
        print(f"  Proceeding with conversion — TFLiteConverter handles this separately.")

    if FP32_PATH.exists():
        print(f"  ✅ FP32 TFLite already exists: {FP32_PATH}")
    else:
        print("  Converting ...", end=" ", flush=True)
        converter = _get_converter(SAVED_MODEL_DIR)
        buf = converter.convert()
        FP32_PATH.write_bytes(buf)
        print(f"✅ Saved FP32 TFLite: {FP32_PATH}  ({len(buf)/1024:.1f} KB)")

    # ══════════════════════════════════════════════════════════════════════════
    # Step 3: Ensure calibration dataset (Speech Commands v0.02 validation)
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print("  Step 3: Ensure calibration dataset")
    print("=" * 60)

    def _list_wavs(folder, need=99999):
        """Return up to `need` WAV file paths from folder (recursive)."""
        out = []
        for root, _, files in os.walk(str(folder)):
            for f in sorted(files):
                if f.lower().endswith(".wav"):
                    out.append(pathlib.Path(root) / f)
                    if len(out) >= need:
                        return out
        return out

    _calib_wavs = _list_wavs(CALIB_WAV_DIR) if CALIB_WAV_DIR.is_dir() else []

    if len(_calib_wavs) >= 10:
        print(f"  ✅ Calibration dataset found: {CALIB_WAV_DIR} ({len(_calib_wavs)} WAVs)")
    else:
        print(f"  ⚠️  Calibration WAVs not found — downloading Speech Commands v0.02 ...")

        # Download via tensorflow_datasets
        import tensorflow_datasets as tfds
        _tfds_dir = pathlib.Path.home() / "data"
        print(f"  Data directory: {_tfds_dir}")

        tfds.load("speech_commands", split="validation",
                  data_dir=str(_tfds_dir), download=True)
        print(f"  ✅ Dataset downloaded")

        # Find validation tfrecord shards
        _pattern = str(_tfds_dir / "speech_commands" / "*" /
                       "speech_commands-validation.tfrecord-*")
        _shards = sorted(_glob.glob(_pattern))
        if not _shards:
            raise FileNotFoundError(f"No validation tfrecord shards found: {_pattern}")
        print(f"  Found {len(_shards)} validation shard(s)")

        # Extract WAV files organised by keyword sub-folder
        CALIB_WAV_DIR.mkdir(parents=True, exist_ok=True)
        _count = 0
        _label_counts = {i: 0 for i in range(12)}

        for raw_record in tqdm(tf.data.TFRecordDataset(_shards),
                               desc="  Extracting WAVs", unit="wav"):
            example = tf.train.Example()
            example.ParseFromString(raw_record.numpy())
            label_idx = example.features.feature['label'].int64_list.value[0]
            audio_int16 = np.array(
                example.features.feature['audio'].int64_list.value, dtype=np.int16)
            label_name = KWS_LABELS[label_idx] if label_idx < len(KWS_LABELS) else f"class_{label_idx}"

            kw_dir = CALIB_WAV_DIR / label_name
            kw_dir.mkdir(exist_ok=True)
            wav_path = kw_dir / f"{label_name}_{_count:05d}.wav"
            with _wave.open(str(wav_path), "w") as wf:
                wf.setnchannels(1)
                wf.setsampwidth(2)
                wf.setframerate(SAMPLE_RATE)
                wf.writeframes(audio_int16.tobytes())

            _label_counts[label_idx] = _label_counts.get(label_idx, 0) + 1
            _count += 1

        print(f"  ✅ Extracted {_count} WAV files to {CALIB_WAV_DIR}")
        for i, name in enumerate(KWS_LABELS):
            print(f"     {name:>10s}: {_label_counts.get(i, 0)}")

        _calib_wavs = _list_wavs(CALIB_WAV_DIR)

    # ══════════════════════════════════════════════════════════════════════════
    # Step 4: SavedModel → TFLite INT8 (with balanced calibration)
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print("  Step 4: Convert SavedModel → TFLite INT8 (with calibration)")
    print("=" * 60)

    if INT8_PATH.exists():
        print(f"  ✅ INT8 TFLite already exists: {INT8_PATH}")
    else:
        # Balanced sampling: pick equal WAVs from each keyword sub-folder
        _subdirs = sorted([
            d for d in CALIB_WAV_DIR.iterdir()
            if d.is_dir()
        ])

        if _subdirs:
            _per_class = max(1, NUM_CALIB_SAMPLES // len(_subdirs))
            _selected = []
            for subdir in _subdirs:
                wavs = sorted(subdir.glob("*.wav"))
                step = max(1, len(wavs) // _per_class)
                _selected.extend(wavs[::step][:_per_class])
            _selected = _selected[:NUM_CALIB_SAMPLES]
            print(f"  Balanced sampling: {_per_class} per class × {len(_subdirs)} classes")
        else:
            _selected = _calib_wavs[:NUM_CALIB_SAMPLES]

        print(f"  Using {len(_selected)} WAVs for calibration ...")

        def _load_and_mfcc(wav_path):
            """Load WAV → normalize → pad/truncate → MFCC features."""
            raw = tf.io.read_file(str(wav_path))
            audio, _ = tf.audio.decode_wav(raw, desired_channels=1)
            audio = tf.squeeze(audio, axis=-1).numpy().astype(np.float32)
            if len(audio) < DESIRED_SAMPLES:
                audio = np.pad(audio, (0, DESIRED_SAMPLES - len(audio)))
            else:
                audio = audio[:DESIRED_SAMPLES]
            peak = np.max(np.abs(audio))
            if peak > 0:
                audio = audio / peak
            # Inline MFCC extraction (same as extract_mfcc in Section 3)
            stfts = tf.signal.stft(audio, frame_length=WINDOW_SIZE_SAMPLES,
                                   frame_step=WINDOW_STRIDE_SAMPLES,
                                   window_fn=tf.signal.hann_window)
            spectrograms = tf.abs(stfts)
            num_bins = stfts.shape[-1]
            mel_matrix = tf.signal.linear_to_mel_weight_matrix(
                NUM_MEL_BINS, num_bins, SAMPLE_RATE, LOWER_EDGE_HZ, UPPER_EDGE_HZ)
            mel_spec = tf.tensordot(spectrograms, mel_matrix, 1)
            log_mel = tf.math.log(mel_spec + 1e-6)
            mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel)
            mfccs = mfccs[..., :DCT_COEFFICIENT_COUNT]
            mfccs = tf.reshape(mfccs, [mfccs.shape[0], DCT_COEFFICIENT_COUNT, 1])
            return mfccs.numpy()

        _pbar = tqdm(total=len(_selected), desc="  Calibrating", unit="wav")

        def _rep_dataset():
            for wp in _selected:
                mfcc = _load_and_mfcc(wp).astype(np.float32)
                _pbar.update(1)
                yield [mfcc[np.newaxis, ...]]   # shape (1, 49, 10, 1)

        print("  Running TFLite quantization ...")
        converter = _get_converter(SAVED_MODEL_DIR)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = _rep_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

        buf = converter.convert()
        _pbar.close()
        INT8_PATH.write_bytes(buf)
        print(f"  ✅ Saved INT8 TFLite: {INT8_PATH}  ({len(buf)/1024:.1f} KB)")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  All models ready!")
    print(f"{'='*60}")
    print(f"  FP32 : {FP32_PATH}  ({FP32_PATH.stat().st_size/1024:.1f} KB)")
    print(f"  INT8 : {INT8_PATH}  ({INT8_PATH.stat().st_size/1024:.1f} KB)")

---
### 🔬 Section 1.1 — Verify Original SavedModel (Baseline Inference)

Before converting to TFLite, run a quick inference using the **original SavedModel**
directly via `tf.saved_model.load()`. This serves as a **ground-truth baseline** for
comparing against the FP32 and INT8 TFLite models later.

| What | Details |
|------|---------|
| **Model** | KWS DS-CNN SavedModel (MLCommons Tiny) |
| **API** | `tf.saved_model.load()` → `signatures["serving_default"]` |
| **Input** | MFCC features `[1, 49, 10, 1]`, float32 |
| **Output** | `[1, 12]` — softmax probabilities (12 keyword classes) |


In [ ]:
import numpy as np
from tqdm.auto import tqdm

# ── Load the SavedModel using tf.compat.v1 (works with TF1-style models) ─────
SAVED_MODEL_DIR = TFLITE_DIR / "kws_ref_model"
if not SAVED_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"❌ SavedModel not found at {SAVED_MODEL_DIR}\n"
        f"   Run Section 1 first to download the SavedModel."
    )

# The MLCommons Tiny KWS SavedModel is TF1-style.
# tf.saved_model.load() fails on TF 2.16+ with:
#   AttributeError: '_UserObject' object has no attribute 'add_slot'
# Solution: use tf.compat.v1 session-based loading.

tf.compat.v1.reset_default_graph()
_sess = tf.compat.v1.Session()
_meta = tf.compat.v1.saved_model.loader.load(
    _sess, [tf.saved_model.SERVING], str(SAVED_MODEL_DIR)
)
_sig = _meta.signature_def["serving_default"]
_input_name = list(_sig.inputs.values())[0].name
_output_name = list(_sig.outputs.values())[0].name
_input_tensor = _sess.graph.get_tensor_by_name(_input_name)
_output_tensor = _sess.graph.get_tensor_by_name(_output_name)

print(f"SavedModel loaded (tf.compat.v1): {SAVED_MODEL_DIR}")
print(f"  Input tensor  : {_input_name}  shape={_input_tensor.shape}")
print(f"  Output tensor : {_output_name}  shape={_output_tensor.shape}\n")

# ── MFCC extraction (inline — same as Section 3) ─────────────────────────────
def _extract_mfcc_baseline(audio):
    stfts = tf.signal.stft(audio, frame_length=WINDOW_SIZE_SAMPLES,
                           frame_step=WINDOW_STRIDE_SAMPLES,
                           window_fn=tf.signal.hann_window)
    spectrograms = tf.abs(stfts)
    num_bins = stfts.shape[-1]
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        NUM_MEL_BINS, num_bins, SAMPLE_RATE, LOWER_EDGE_HZ, UPPER_EDGE_HZ)
    mel_spec = tf.tensordot(spectrograms, mel_matrix, 1)
    log_mel = tf.math.log(mel_spec + 1e-6)
    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel)
    mfccs = mfccs[..., :DCT_COEFFICIENT_COUNT]
    mfccs = tf.reshape(mfccs, [mfccs.shape[0], DCT_COEFFICIENT_COUNT, 1])
    return mfccs.numpy()

def _load_wav_baseline(wav_path):
    raw = tf.io.read_file(str(wav_path))
    audio, _ = tf.audio.decode_wav(raw, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1).numpy().astype(np.float32)
    if len(audio) < DESIRED_SAMPLES:
        audio = np.pad(audio, (0, DESIRED_SAMPLES - len(audio)))
    else:
        audio = audio[:DESIRED_SAMPLES]
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

# ── Run on test WAVs ──────────────────────────────────────────────────────────
test_wavs = sorted(TEST_DATA.glob("*.wav"))
print(f"{'='*60}")
print(f"  Original SavedModel — KWS DS-CNN Baseline")
print(f"{'='*60}")
print(f"  Found {len(test_wavs)} test WAV files\n")

print(f"  {'WAV file':<28s} {'Prediction':<15s} {'Confidence':>10s}")
print(f"  {'─'*58}")

for wav in tqdm(test_wavs, desc="  SavedModel inference", unit="wav"):
    audio = _load_wav_baseline(wav)
    mfcc = _extract_mfcc_baseline(audio)
    input_data = mfcc[np.newaxis].astype(np.float32)  # (1, 49, 10, 1)

    probs = _sess.run(_output_tensor, feed_dict={_input_tensor: input_data}).squeeze()

    # Apply softmax if needed
    if not (np.all(probs >= 0) and np.isclose(probs.sum(), 1.0, atol=0.05)):
        e = np.exp(probs - np.max(probs))
        probs = e / e.sum()

    pred_idx = np.argmax(probs)
    pred_label = KWS_LABELS[pred_idx]
    pred_conf = probs[pred_idx]
    print(f"  {wav.name:<28s} {pred_label:<15s} {pred_conf:>9.2%}")

_sess.close()
print(f"\n✅ Original SavedModel verified — predictions look correct.")
print(f"   This serves as the ground-truth baseline for TFLite comparison.")

---
## 🔍 Section 2 — Verify Pre-built Models

The TFLite FP32 and INT8 models are **pre-built and included in the repository**.
This cell verifies the models exist and inspects their I/O shapes and quantization parameters.


In [ ]:
# ── TFLite FP32 ───────────────────────────────────────────────────────────────
print(f"{'='*55}")
print(f"  TFLite FP32 — {FP32_PATH.name}")
print(f"{'='*55}")
interp_fp32 = tf.lite.Interpreter(model_path=str(FP32_PATH))
interp_fp32.allocate_tensors()
inp_f = interp_fp32.get_input_details()[0]
out_f = interp_fp32.get_output_details()[0]
print(f"  Input  : shape={inp_f['shape']}  dtype={inp_f['dtype'].__name__}")
print(f"  Output : shape={out_f['shape']}  dtype={out_f['dtype'].__name__}")
print(f"  Size   : {FP32_PATH.stat().st_size / 1024:.1f} KB")

# ── TFLite INT8 ───────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  TFLite INT8 — {INT8_PATH.name}")
print(f"{'='*55}")
interp_int8 = tf.lite.Interpreter(model_path=str(INT8_PATH))
interp_int8.allocate_tensors()
inp_i = interp_int8.get_input_details()[0]
out_i = interp_int8.get_output_details()[0]
print(f"  Input  : shape={inp_i['shape']}  dtype={inp_i['dtype'].__name__}")
print(f"  Output : shape={out_i['shape']}  dtype={out_i['dtype'].__name__}")
print(f"  Input  quant: scale={inp_i['quantization_parameters']['scales'][0]:.6f}  zp={inp_i['quantization_parameters']['zero_points'][0]}")
print(f"  Output quant: scale={out_i['quantization_parameters']['scales'][0]:.8f}  zp={out_i['quantization_parameters']['zero_points'][0]}")
print(f"  Size   : {INT8_PATH.stat().st_size / 1024:.1f} KB")

print(f"\n✅ Both models verified — input [1, 49, 10, 1], output [1, 12]")

---
## 📐 Section 3 — Preprocessing & Postprocessing Helpers

The KWS DS-CNN model does **not** take raw audio as input. The audio must first be
converted to MFCC (Mel-Frequency Cepstral Coefficients) features, and the raw model
output must be decoded into class probabilities.

### Preprocessing Pipeline

| Step | Operation | Details |
|------|-----------|---------|
| 1 | **Load WAV** | Read 16 kHz mono WAV file |
| 2 | **Pad / truncate** | Ensure exactly 16,000 samples (1 second) |
| 3 | **Normalise** | Scale to [-1, 1] by dividing by peak amplitude |
| 4 | **STFT** | Short-Time Fourier Transform (frame=480, hop=320, Hann window) → 49 frames |
| 5 | **Mel filterbank** | Project spectrogram onto 40 mel bins (20–4000 Hz) |
| 6 | **Log-mel** | `log(mel + 1e-6)` for numerical stability |
| 7 | **DCT** | Discrete Cosine Transform → keep first 10 coefficients |
| 8 | **Reshape** | `[49, 10]` → `[49, 10, 1]` (add channel dimension) |
| 9 | **Quantize** (INT8 only) | `int8_val = clip(round(float_val / scale + zp), -128, 127)` |

```
16 kHz mono WAV  (1 second = 16,000 samples)
     │
     ▼  Pad/truncate to 16,000 samples
     │
     ▼  Normalise to [-1, 1]
     │
     ▼  STFT (frame=480, hop=320, Hann) → 49 frames × 241 bins
     │
     ▼  Mel filterbank (40 bins, 20–4000 Hz)
     │
     ▼  Log → DCT (keep 10 coefficients)
     │
     ▼  Reshape to [1, 49, 10, 1]
     │
     ▼  (INT8: quantize using input scale/zp)
  Model input
```

### Postprocessing Pipeline

| Step | Operation | Details |
|------|-----------|---------|
| 1 | **Dequantize** (INT8 only) | `float_val = (int8_val - zp) * scale` |
| 2 | **Softmax** | Convert logits to probabilities: `exp(x) / sum(exp(x))` |
| 3 | **Argmax** | Select class with highest probability |
| 4 | **Label lookup** | Map class index → keyword label |

```
Raw output [1, 12]  (logits or int8 values)
     │
     ▼  (INT8: dequantize → float32)
     │
     ▼  Softmax → probabilities [1, 12]
     │
     ▼  argmax → class index
     │
     ▼  KWS_LABELS[index]
  Predicted keyword: "yes", "no", "up", "down", etc.
```

### MFCC Parameters (must match training)

| Parameter | Value | Description |
|-----------|-------|-------------|
| Sample rate | 16,000 Hz | Mono |
| Clip length | 1.0 s | 16,000 samples |
| STFT frame | 480 samples (30 ms) | Hann window |
| STFT stride | 320 samples (20 ms) | → 49 frames |
| Mel bins | 40 | 20–4,000 Hz |
| MFCC coefficients | 10 | DCT of log-mel |
| Model input shape | `[1, 49, 10, 1]` | batch × frames × coefficients × channel |


In [ ]:
# ── Audio loading (preprocessing step 1–3) ────────────────────────────────────
def load_wav(wav_path):
    """Load a WAV file → float32 samples normalised to [-1, 1], padded/truncated to 1 s."""
    raw = tf.io.read_file(str(wav_path))
    audio, sr = tf.audio.decode_wav(raw, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1).numpy().astype(np.float32)

    # Pad or truncate to 16,000 samples
    if len(audio) < DESIRED_SAMPLES:
        audio = np.pad(audio, (0, DESIRED_SAMPLES - len(audio)))
    else:
        audio = audio[:DESIRED_SAMPLES]

    # Normalise to [-1, 1]
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio


# ── MFCC extraction (preprocessing steps 4–8) ────────────────────────────────
def extract_mfcc(audio):
    """Extract MFCC features from 1-second float32 audio.

    Pipeline: STFT → Mel filterbank → Log → DCT → keep 10 coefficients
    Returns: np.ndarray, shape (49, 10, 1), float32
    """
    stfts = tf.signal.stft(
        audio,
        frame_length=WINDOW_SIZE_SAMPLES,
        frame_step=WINDOW_STRIDE_SAMPLES,
        window_fn=tf.signal.hann_window,
    )
    spectrograms = tf.abs(stfts)
    num_bins = stfts.shape[-1]

    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        NUM_MEL_BINS, num_bins, SAMPLE_RATE, LOWER_EDGE_HZ, UPPER_EDGE_HZ,
    )
    mel_spec = tf.tensordot(spectrograms, mel_matrix, 1)
    log_mel  = tf.math.log(mel_spec + 1e-6)
    mfccs    = tf.signal.mfccs_from_log_mel_spectrograms(log_mel)
    mfccs    = mfccs[..., :DCT_COEFFICIENT_COUNT]
    mfccs    = tf.reshape(mfccs, [mfccs.shape[0], DCT_COEFFICIENT_COUNT, 1])
    return mfccs.numpy()


# ── Softmax (postprocessing step 2) ──────────────────────────────────────────
def softmax(x):
    """Convert logits to probabilities."""
    e = np.exp(x - np.max(x))
    return e / e.sum(axis=-1, keepdims=True)


# ── KWS inference helper (full pre + post) ───────────────────────────────────
def run_kws_inference(model_path, mfcc_features):
    """Run KWS TFLite inference on pre-extracted MFCC features.

    Handles INT8 quantization/dequantization automatically.
    Returns: np.ndarray of shape (12,) — probabilities.
    """
    interp = tf.lite.Interpreter(model_path=str(model_path))
    interp.allocate_tensors()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    inp_arr = mfcc_features[np.newaxis].astype(np.float32)  # (1, 49, 10, 1)

    # Preprocessing step 9: quantize for INT8
    if inp_det["dtype"] == np.int8:
        scale, zp = inp_det["quantization"]
        inp_arr = np.clip(np.round(inp_arr / scale + zp), -128, 127).astype(np.int8)

    interp.set_tensor(inp_det["index"], inp_arr)
    interp.invoke()
    raw = interp.get_tensor(out_det["index"]).astype(np.float32).squeeze()

    # Postprocessing step 1: dequantize INT8 output
    if out_det["dtype"] != np.float32:
        scale, zp = out_det["quantization"]
        raw = (raw - zp) * scale

    # Postprocessing step 2: softmax if needed
    if np.all(raw >= 0) and np.isclose(raw.sum(), 1.0, atol=0.05):
        return raw
    return softmax(raw)


# ── Helper: find any available WAV for testing ────────────────────────────────
def _find_test_wav():
    """Return a single WAV path for quick sanity check.

    Priority: sample_audio/test_yes.wav → any WAV in sample_audio/ → any calib WAV.
    """
    candidate = TEST_DATA / "test_yes.wav"
    if candidate.exists():
        return candidate
    # Fallback: first WAV in sample_audio/
    if TEST_DATA.is_dir():
        wavs = sorted(TEST_DATA.glob("*.wav"))
        if wavs:
            return wavs[0]
    # Fallback: first calibration WAV
    _calib_dir = TFLITE_DIR / "calib_wavs"
    if _calib_dir.is_dir():
        for root, _, files in os.walk(str(_calib_dir)):
            for f in sorted(files):
                if f.lower().endswith(".wav"):
                    return pathlib.Path(root) / f
    return None


# ── Quick test on a sample WAV ────────────────────────────────────────────────
test_wav = _find_test_wav()
if test_wav is None:
    raise FileNotFoundError(
        f"❌ No WAV files found for quick test.\n"
        f"   Looked in: {TEST_DATA} and {TFLITE_DIR / 'calib_wavs'}\n"
        f"   Run Section 1 first to download test data."
    )
print(f"Test WAV      : {test_wav.name}")
audio = load_wav(test_wav)
mfcc  = extract_mfcc(audio)
print(f"Audio samples : {len(audio)}  (expected {DESIRED_SAMPLES})")
print(f"MFCC shape    : {mfcc.shape}  (expected [49, 10, 1])")
print(f"MFCC range    : [{mfcc.min():.2f}, {mfcc.max():.2f}]")
print(f"\n✅ Preprocessing & postprocessing helpers ready")


---
## 🔥 Section 4 — Quick Inference Test (TFLite FP32 vs INT8)

Run KWS inference on several test WAV files using both FP32 and INT8 TFLite models
to verify the pipeline before MERA compilation.


In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

MAX_EVAL_WAVS = 24   # cap evaluation set for fast, stable notebook runs

# ── Collect evaluation WAVs (sample_audio preferred, calibration fallback) ───────
def _collect_eval_wavs(max_wavs=MAX_EVAL_WAVS):
    """Return a list of WAV paths for evaluation.

    Priority: sample_audio/*.wav → calibration WAVs (balanced, capped).
    """
    wavs = sorted(TEST_DATA.glob("*.wav")) if TEST_DATA.is_dir() else []
    if wavs:
        return wavs[:max_wavs]
    # Fallback: sample from calibration WAVs (balanced across sub-folders)
    _calib_dir = TFLITE_DIR / "calib_wavs"
    if _calib_dir.is_dir():
        subdirs = sorted([d for d in _calib_dir.iterdir() if d.is_dir()])
        if subdirs:
            per_class = max(1, max_wavs // len(subdirs))
            out = []
            for sd in subdirs:
                out.extend(sorted(sd.glob("*.wav"))[:per_class])
            return out[:max_wavs]
        # Flat folder
        return sorted(_calib_dir.glob("*.wav"))[:max_wavs]
    return []

test_wavs = _collect_eval_wavs()
if not test_wavs:
    raise FileNotFoundError(
        f"❌ No WAV files found for evaluation.\n"
        f"   Looked in: {TEST_DATA} and {TFLITE_DIR / 'calib_wavs'}\n"
        f"   Run Section 1 first to download/build models and test data."
    )

_src = "sample_audio" if TEST_DATA.is_dir() and sorted(TEST_DATA.glob("*.wav")) else "calibration (fallback)"
print(f"Evaluation source : {_src}")
print(f"Found {len(test_wavs)} WAV files (capped at {MAX_EVAL_WAVS})\n")

print(f"{'WAV file':<30s} {'FP32 prediction':<20s} {'INT8 prediction':<20s}  Match")
print("─" * 95)

matches = 0
for wav in tqdm(test_wavs, desc="FP32/INT8 inference", unit="wav"):
    audio = load_wav(wav)
    mfcc  = extract_mfcc(audio)

    probs_fp32 = run_kws_inference(FP32_PATH, mfcc)
    probs_int8 = run_kws_inference(INT8_PATH, mfcc)

    pred_fp32 = KWS_LABELS[np.argmax(probs_fp32)]
    pred_int8 = KWS_LABELS[np.argmax(probs_int8)]
    conf_fp32 = np.max(probs_fp32)
    conf_int8 = np.max(probs_int8)
    match = "✅" if pred_fp32 == pred_int8 else "❌"
    if pred_fp32 == pred_int8:
        matches += 1

    tqdm.write(f"  {wav.name:<28s} {pred_fp32:<12s} ({conf_fp32:.2f})   {pred_int8:<12s} ({conf_int8:.2f})   {match}")

print(f"\n✅ FP32/INT8 agreement: {matches}/{len(test_wavs)} ({100*matches/len(test_wavs):.0f}%)")


In [ ]:
# ── Visualise KWS results — one row per test WAV ─────────────────────────────
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from tqdm.auto import tqdm

# Re-use the evaluation WAV list from Section 4 (already capped + fallback)
if not test_wavs:
    raise FileNotFoundError(
        "❌ No evaluation WAVs available — run the previous cell first."
    )

n_samples = len(test_wavs)

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — All test samples: Waveform | MFCC | Probability bar chart
# ═══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 2.8 * n_samples))
gs = GridSpec(n_samples, 3, figure=fig, wspace=0.30, hspace=0.50,
              width_ratios=[1, 1, 1.3])

for row, wav in enumerate(tqdm(test_wavs, desc="Building plots", unit="wav")):
    audio = load_wav(wav)
    mfcc  = extract_mfcc(audio)
    probs_fp32 = run_kws_inference(FP32_PATH, mfcc)
    probs_int8 = run_kws_inference(INT8_PATH, mfcc)
    expected = wav.stem.replace("test_", "").strip("_")

    # ── Col 0: Waveform ──────────────────────────────────────────────────────
    ax0 = fig.add_subplot(gs[row, 0])
    ax0.plot(audio, linewidth=0.4, color="steelblue")
    ax0.set_title(f'{wav.name}  (expected: "{expected}")',
                  fontsize=9, fontweight="bold")
    ax0.set_ylabel("Amp", fontsize=7)
    ax0.tick_params(labelsize=6)
    if row == n_samples - 1:
        ax0.set_xlabel("Sample", fontsize=7)

    # ── Col 1: MFCC spectrogram ──────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[row, 1])
    im = ax1.imshow(mfcc[:, :, 0].T, aspect="auto", origin="lower", cmap="viridis")
    ax1.set_title("MFCC (49×10)", fontsize=9)
    ax1.set_ylabel("Coeff", fontsize=7)
    ax1.tick_params(labelsize=6)
    if row == n_samples - 1:
        ax1.set_xlabel("Time frame", fontsize=7)

    # ── Col 2: Class probability bar chart (log scale) ───────────────────────
    ax2 = fig.add_subplot(gs[row, 2])
    x = np.arange(len(KWS_LABELS))
    w = 0.35
    ax2.bar(x - w/2, probs_fp32, w, label="FP32", alpha=0.85, color="tab:blue")
    ax2.bar(x + w/2, probs_int8, w, label="INT8", alpha=0.85, color="tab:orange")
    ax2.set_yscale("log")
    ax2.set_ylim(1e-4, 2.0)
    ax2.set_xticks(x)
    ax2.set_xticklabels(KWS_LABELS, rotation=45, ha="right", fontsize=7)
    ax2.axhline(0.5, color="grey", ls="--", lw=0.7, alpha=0.5)
    ax2.tick_params(labelsize=6)

    top_fp = np.argmax(probs_fp32)
    top_i8 = np.argmax(probs_int8)
    match_str = "Match" if top_fp == top_i8 else "MISMATCH"
    ax2.set_title(
        f'FP32: {KWS_LABELS[top_fp]} ({probs_fp32[top_fp]:.2f})   '
        f'INT8: {KWS_LABELS[top_i8]} ({probs_int8[top_i8]:.2f})   [{match_str}]',
        fontsize=8, fontweight="bold")

    # Annotate top bars
    ax2.annotate(f"{probs_fp32[top_fp]:.2f}",
                 xy=(top_fp - w/2, probs_fp32[top_fp]),
                 ha="center", va="bottom", fontsize=6,
                 color="tab:blue", fontweight="bold")
    ax2.annotate(f"{probs_int8[top_i8]:.2f}",
                 xy=(top_i8 + w/2, probs_int8[top_i8]),
                 ha="center", va="bottom", fontsize=6,
                 color="tab:orange", fontweight="bold")
    if row == 0:
        ax2.legend(fontsize=7, loc="upper left")

fig.suptitle("KWS Inference — All Test Samples  (Waveform → MFCC → Class Probabilities)",
             fontsize=14, fontweight="bold", y=1.002)
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Summary heatmap (compact overview)
# ═══════════════════════════════════════════════════════════════════════════════
print("Building heatmap data ...")
wav_names = [w.stem.replace("test_", "").strip("_") for w in test_wavs]
fp32_matrix = np.stack([run_kws_inference(FP32_PATH, extract_mfcc(load_wav(w)))
                        for w in tqdm(test_wavs, desc="Heatmap FP32", unit="wav")])
int8_matrix = np.stack([run_kws_inference(INT8_PATH, extract_mfcc(load_wav(w)))
                        for w in tqdm(test_wavs, desc="Heatmap INT8", unit="wav")])

fig2, (ax_fp, ax_i8) = plt.subplots(1, 2, figsize=(16, max(3.5, 0.45 * n_samples + 2)),
                                    gridspec_kw={"wspace": 0.4})
fig2.suptitle("KWS Prediction Heatmap — All Test WAVs", fontsize=13, fontweight="bold")

for ax, mat, title in [(ax_fp, fp32_matrix, "FP32"), (ax_i8, int8_matrix, "INT8")]:
    im = ax.imshow(mat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_yticks(range(len(wav_names)))
    ax.set_yticklabels(wav_names, fontsize=8)
    ax.set_xticks(range(len(KWS_LABELS)))
    ax.set_xticklabels(KWS_LABELS, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Predicted class", fontsize=9)
    ax.set_ylabel("Test sample", fontsize=9)
    ax.set_title(title, fontsize=11, fontweight="bold")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if v > 0.01:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if v > 0.5 else "black",
                        fontweight="bold" if v > 0.5 else "normal")
    fig2.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Probability")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
agree = sum(1 for i in range(n_samples)
            if np.argmax(fp32_matrix[i]) == np.argmax(int8_matrix[i]))
print(f"\n✅ FP32/INT8 agree on {agree}/{n_samples} samples")
print("   Bar charts: each bar = probability of that class (log scale)")
print("   Heatmap: bright red on diagonal = correct high-confidence prediction")


---
## 🏗️ Section 5 — MERA Compilation

Compiles the KWS model to MCU C-code using the **MERA (RUHMI)** compiler.

---

### 5.1 — CPU + NPU Compilation

Compile for **Ethos-U55 NPU**(Requires INT8 model)


```
[CPU+NPU]  kws_ref_model_INT8.tflite
            │
            ▼
       python mcu_compile.py model_INT8.tflite embedded_c/src_mcu_npu/ --npu
            └── Ethos-U55 NPU C source files
```


In [ ]:
import numpy as np
import pathlib

COMPILE_PY = REPO_ROOT / "ruhmi_tools" / "mcu_compile.py"
NPU_OUT    = EMBED_DIR / "src_mcu_npu"

if not COMPILE_PY.exists():
    raise FileNotFoundError(f"❌ MERA compile script not found: {COMPILE_PY}")
if not INT8_PATH.exists():
    raise FileNotFoundError(f"❌ INT8 TFLite model not found: {INT8_PATH}\n   Run Section 1 first.")

# ── NPU compile (INT8 TFLite → Ethos-U55 NPU C) ────────────────────────────
print("\n" + "=" * 60)
print("NPU Compilation")
print("=" * 60)
print(f"  Input : {INT8_PATH}")
!python "{COMPILE_PY}" "{INT8_PATH}" "{NPU_OUT}" --npu

print(f"✅ NPU output → {NPU_OUT}")


---
### 5.2 — CPU-Only Compilation

Compile for **Cortex-M CPU (CMSIS-NN)** with `--host-evaluate` to also build
a `py_compute.so` shared library for host-side verification.

```
kws_ref_model_INT8.tflite
        │
        ▼
python mcu_compile.py model_INT8.tflite src_mcu/ --cpu --host-evaluate
        └── CMSIS-NN C files  →  embedded_c/src_mcu/
        └── py_compute.so     →  host verification library
```

In [ ]:
import shutil, os, subprocess, sys

COMPILE_PY = REPO_ROOT / "ruhmi_tools" / "mcu_compile.py"
CPU_OUT    = EMBED_DIR / "src_mcu"

if not COMPILE_PY.exists():
    raise FileNotFoundError(f"❌ MERA compile script not found: {COMPILE_PY}")
if not INT8_PATH.exists():
    raise FileNotFoundError(f"❌ INT8 TFLite model not found: {INT8_PATH}\n   Run Section 1 first.")

# ── CPU compile (INT8 TFLite → CMSIS-NN C) ───────────────────────────────────
print("=" * 60)
print("  CPU-Only Compilation (with host evaluation)")
print("=" * 60)
print(f"  Input : {INT8_PATH}")

# ⚠️  The CMake host-evaluate build requires a C++ compiler.
env = os.environ.copy()
if not shutil.which("g++"):
    for _ver in ["11", "12", "13", "14"]:
        _gpp = shutil.which(f"g++-{_ver}")
        _gcc = shutil.which(f"gcc-{_ver}")
        if _gpp and _gcc:
            env["CXX"] = _gpp
            env["CC"]  = _gcc
            print(f"  ⚠️  g++ not in PATH — using: CXX={_gpp}, CC={_gcc}")
            break
    else:
        print("  ❌ WARNING: No C++ compiler found! Install: sudo apt install g++-11")

# --host-evaluate builds py_compute.so via CMake so we can verify
# the compiled model on the host before flashing to the board.
result = subprocess.run(
    [sys.executable, str(COMPILE_PY), str(INT8_PATH), str(CPU_OUT), "--cpu", "--host-evaluate"],
    env=env,
)

print(f"\n✅ CPU output → {CPU_OUT}")
if result.returncode != 0:
    print(f"  ⚠️  mcu_compile exited with code {result.returncode}")

---
## ✅ Section 6 — Verify MERA-Compiled Model (py_compute Inference)

Since we used `--host-evaluate` in Section 5.2, the `mcu_compile.py` script has
already built the MERA-generated C source into a `py_compute` shared library
(via CMake with `-DBUILD_PY_BINDINGS=ON`).

This section loads the pre-built `py_compute.so` and runs inference on the test WAV
files to verify that the compiled model produces correct keyword predictions
— matching the TFLite INT8 results from Section 4.

> The `py_compute` library is a Python-callable wrapper around the same C-code
> that will run on the RA8P1 board, so matching results here confirms the
> MERA compilation preserved model accuracy.

### ⚠️ Host Build Requirements (for `--host-evaluate`)

| Requirement | Minimum Version | Check Command | Install / Fix |
|-------------|----------------|---------------|---------------|
| **CMake** | ≥ 3.24 | `cmake --version` | `pip install --upgrade cmake` |
| **C++ compiler** (g++) | g++-11 recommended | `g++ --version` | `sudo apt install g++-11` |
| **build-essential** | — | `dpkg -l build-essential` | `sudo apt install build-essential` |

> 💡 **Common issues:**
> - **CMake too old** — Ubuntu 22.04 ships CMake 3.22, but the build needs ≥ 3.24.
>   Upgrade via pip: `pip install --upgrade cmake`
> - **g++ not in PATH** — The cell auto-detects `g++-11`. If that fails, set manually:
>   ```bash
>   export CXX=/usr/bin/g++-11
>   export CC=/usr/bin/gcc-11
>   ```
> - **pybind11** is fetched automatically by CMake — no manual install needed.

In [ ]:
import subprocess, pickle, sys
import numpy as np
import tensorflow as tf

# ── Locate the pre-built py_compute.so ──────────────────────────────────────
CPU_BUILD_DIR = EMBED_DIR / "src_mcu"
py_compute_candidates = (
    list(CPU_BUILD_DIR.rglob("py_compute*.so"))
    + list(CPU_BUILD_DIR.rglob("py_compute*.pyd"))
)

if not py_compute_candidates:
    raise FileNotFoundError(
        f"❌ py_compute shared library not found in {CPU_BUILD_DIR}\n"
        f"   Make sure Section 5.2 ran with --host-evaluate."
    )

py_compute_dir = str(py_compute_candidates[0].parent)
print(f"✅ Found py_compute: {py_compute_candidates[0]}")

# ── Read INT8 quantization parameters ────────────────────────────────────────
interp_q = tf.lite.Interpreter(model_path=str(INT8_PATH))
interp_q.allocate_tensors()
_inp_det = interp_q.get_input_details()[0]
_out_det = interp_q.get_output_details()[0]
_inp_qp  = _inp_det["quantization_parameters"]
_out_qp  = _out_det["quantization_parameters"]
quantized_io = {
    "input_scale":  float(_inp_qp["scales"][0]),
    "input_zp":     int(_inp_qp["zero_points"][0]),
    "output_scale": float(_out_qp["scales"][0]),
    "output_zp":    int(_out_qp["zero_points"][0]),
}
del interp_q
print(f"INT8 quant params: inp_scale={quantized_io['input_scale']:.6f}  inp_zp={quantized_io['input_zp']}")
print(f"                   out_scale={quantized_io['output_scale']:.6f}  out_zp={quantized_io['output_zp']}")

# ── Subprocess inference (avoids TF / py_compute symbol conflicts) ───────────
def run_mera_audio_subprocess(py_compute_dir, features_list, quantized_io):
    config_file = "/tmp/mera_audio_nb_config.pkl"
    result_file = "/tmp/mera_audio_nb_results.pkl"
    with open(config_file, "wb") as f:
        pickle.dump({
            "py_compute_dir": py_compute_dir,
            "features_list":  features_list,
            "quantized_io":   quantized_io,
        }, f)

    script = f"""
import sys, pickle
import numpy as np

with open("{config_file}", "rb") as f:
    cfg = pickle.load(f)

sys.path.insert(0, cfg["py_compute_dir"])
import py_compute as c

qio     = cfg["quantized_io"]
results = []
for feat in cfg["features_list"]:
    blob = np.array(feat, dtype=np.float32)
    while blob.ndim < 4:
        blob = blob[np.newaxis, ...]
    if qio:
        blob = np.clip(
            np.round(blob / qio["input_scale"] + qio["input_zp"]),
            -128, 127
        ).astype(np.int8)
    raw = c.compute(blob)[0]
    if qio:
        raw = (raw.astype(np.float32) - qio["output_zp"]) * qio["output_scale"]
    elif raw.dtype != np.float32:
        raw = raw.astype(np.float32)
    results.append(raw)

with open("{result_file}", "wb") as f:
    pickle.dump(results, f)
"""
    proc = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  ❌ Subprocess failed:\n{proc.stderr[:2000]}")
        return None
    with open(result_file, "rb") as f:
        return pickle.load(f)

# ── Prepare test features using Section 3 helpers (load_wav + extract_mfcc) ─
# TEST_DATA is defined in the config cell as MODEL_DIR / "sample_audio"
# It contains one .wav file per keyword class (e.g. test_yes.wav, test_no.wav)
eval_wavs = sorted(TEST_DATA.glob("*.wav")) if TEST_DATA.exists() else []
if not eval_wavs:
    raise FileNotFoundError(
        f"❌ No test WAV files found in {TEST_DATA}\n"
        f"   Run Section 1 first to download test data."
    )
print(f"\nFound {len(eval_wavs)} test WAV files in {TEST_DATA}")

features_list = []
labels        = []
for wav_path in eval_wavs:
    audio = load_wav(wav_path)
    mfcc  = extract_mfcc(audio)
    features_list.append(mfcc)
    labels.append(wav_path.stem.replace("test_", ""))

# ── TFLite INT8 reference ────────────────────────────────────────────────────
interp_sec6 = tf.lite.Interpreter(model_path=str(INT8_PATH))
interp_sec6.allocate_tensors()
inp_d = interp_sec6.get_input_details()[0]
out_d = interp_sec6.get_output_details()[0]

tflite_results = []
for feat in features_list:
    blob = feat.copy()
    if blob.ndim < len(inp_d["shape"]):
        blob = blob[np.newaxis, ...]
    if inp_d["dtype"] == np.int8:
        s, zp = inp_d["quantization_parameters"]["scales"][0], inp_d["quantization_parameters"]["zero_points"][0]
        blob = np.clip(np.round(blob / s + zp), -128, 127).astype(np.int8)
    interp_sec6.set_tensor(inp_d["index"], blob)
    interp_sec6.invoke()
    raw = interp_sec6.get_tensor(out_d["index"])
    if raw.dtype != np.float32:
        s, zp = out_d["quantization_parameters"]["scales"][0], out_d["quantization_parameters"]["zero_points"][0]
        raw = (raw.astype(np.float32) - zp) * s
    tflite_results.append(raw[0])
del interp_sec6

# ── MERA py_compute ──────────────────────────────────────────────────────────
mera_results = run_mera_audio_subprocess(py_compute_dir, features_list, quantized_io)
if mera_results is None:
    raise RuntimeError("MERA inference failed — see error above.")

# ── Compare ──────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  TFLite INT8 vs MERA CPU — KWS Prediction Comparison")
print(f"{'='*65}")
print(f"  {'Test sample':<20s} {'TFLite':<14s} {'MERA':<14s} Match")
print(f"  {'─'*58}")

match_count = 0
for i, lbl in enumerate(labels):
    tfl_idx   = int(np.argmax(tflite_results[i]))
    mer_idx   = int(np.argmax(mera_results[i].flatten()))
    tfl_label = KWS_LABELS[tfl_idx] if tfl_idx < len(KWS_LABELS) else str(tfl_idx)
    mer_label = KWS_LABELS[mer_idx] if mer_idx < len(KWS_LABELS) else str(mer_idx)
    ok        = tfl_idx == mer_idx
    match_count += ok
    print(f"  {lbl:<20s} {tfl_label:<14s} {mer_label:<14s} {'✅' if ok else '❌'}")

print(f"\n  Agreement: {match_count}/{len(labels)} ({100*match_count/len(labels):.0f}%)")
print(f"\n{'='*65}")
print(f"  ✅ MERA CPU model verification complete!")
print(f"{'='*65}")

---
### 📄 Understanding the Generated C Files

MERA produces C source files in `embedded_c/src_mcu_npu/` (NPU) or `embedded_c/src_mcu/` (CPU).
The key files are listed below:

| File | Purpose | Target |
|------|---------|--------|
| `model.c / model.h` | Top-level model entry point — orchestrates subgraph execution. | NPU only |
| `hal_entry.c` | Hardware Abstraction Layer — board-specific init (clocks, Ethos-U driver). | NPU only |
| `sub_XXXX_model_data.c/.h` | Vela-compiled command stream & weight data for NPU subgraphs. | NPU only |
| `sub_XXXX_tensors.c/.h` | Tensor arena allocation for NPU subgraphs. | NPU only |
| `sub_XXXX_command_stream.c/.h` | Ethos-U command stream blobs (compiled by Vela). | NPU only |
| `sub_XXXX_invoke.c/.h` | Invoke helpers that feed command streams to the Ethos-U driver. | NPU only |
| `compute_sub_0000.c/.h` | Main inference graph — kernel calls for each layer (NPU-accelerated or CMSIS-NN). | Both |
| `kernel_library_int.c/.h` | INT8/INT16 kernel implementations (Ethos-U offload or CMSIS-NN). | Both |
| `kernel_library_utils.c/.h` | Shared helpers — memory layout, tensor management, dequantization. | Both |

---

## 🎉 You're Done!

You have successfully completed the full KWS pipeline from a pretrained DS-CNN model to MCU-ready C source files:

```
✅  Setup       — Configuration
✅  Section 1   — Download & build models from scratch (optional)
    └─ 1.1 — Verify original SavedModel (baseline inference)
✅  Section 2   — Verify pre-built TFLite models
✅  Section 3   — KWS preprocessing & postprocessing helpers
✅  Section 4   — Quick inference test (SavedModel vs FP32 vs INT8)
✅  Section 5   — MERA compilation → embedded C source files
    ├─ 5.1  NPU compilation (Ethos-U55)
    └─ 5.2  CPU-only compilation (CMSIS-NN) + host evaluation build
✅  Section 6   — Verify MERA-compiled CPU model (py_compute inference)
    └─ Compare keyword predictions vs TFLite INT8 reference
```

The generated C files in `embedded_c/src_mcu/` (and `src_mcu_npu/`) are ready to be
integrated into your **e² studio** project and flashed to the **Renesas RA8P1** board.

---